# Nemotron Reasoning Challenge — LoRA Training

**Workflow:**
1. Setup — install deps, mount Drive, clone repo
2. Baseline sample — ~500 problems to estimate Nano accuracy per category
3. Train LoRA — SFT on formatted training data
4. Eval — run evaluator on val set
5. Submit — package adapter.zip for Kaggle

**GPU**: Requires A100 (Colab Pro). Runtime → Change runtime type → A100.

**Before running**: Upload `train_formatted.jsonl` and `val_formatted.jsonl` to Google Drive.

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────
import subprocess

# Install dependencies
subprocess.run(["pip", "install", "-q",
    "vllm==0.4.3",
    "peft==0.11.1",
    "transformers==4.42.0",
    "trl==0.9.4",
    "datasets==2.19.0",
    "accelerate==0.30.0",
    "bitsandbytes==0.43.1",
], check=True)

print("Dependencies installed.")

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# Set your Drive path where project files live
DRIVE_PATH = "/content/drive/MyDrive/Nemotron"  # adjust if needed
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Drive mounted. Project path: {DRIVE_PATH}")

In [ ]:
# ── Clone/copy pipeline code ───────────────────────────────────────
# Option A: Clone from GitHub (if repo is pushed)
# !git clone https://github.com/YOUR_USERNAME/nemotron-challenge /content/nemotron
# import sys; sys.path.insert(0, '/content/nemotron')

# Option B: Copy pipeline/ from Drive
import shutil, sys
shutil.copytree(f"{DRIVE_PATH}/pipeline", "/content/pipeline", dirs_exist_ok=True)
sys.path.insert(0, "/content")

from pipeline.utils import load_problems, verify, extract_final_answer, stratified_sample
from pipeline.evaluator import Evaluator, stratified_sample
print("Pipeline code loaded.")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────
import os

# Model
MODEL_PATH = "/content/drive/MyDrive/Models/nemotron-3-nano-30b-a3b-bf16"
# Or download from HuggingFace:
# MODEL_PATH = "nvidia/Nemotron-3-Nano-30B-Instruct"  # adjust to actual HF id

# Data
TRAIN_JSONL = f"{DRIVE_PATH}/train_formatted.jsonl"
VAL_JSONL   = f"{DRIVE_PATH}/val_formatted.jsonl"
TRAIN_CSV   = f"{DRIVE_PATH}/train.csv"

# Output
ADAPTER_DIR = "/content/lora_adapter"
RESULTS_DIR = "/content/results"
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# LoRA config — [VERIFIED] from competition demo notebook
LORA_RANK    = 32
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
LORA_TARGETS = ["in_proj", "out_proj", "up_proj", "down_proj"]

# Training
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3
BATCH_SIZE    = 2      # Per-device (A100 40GB)
GRAD_ACCUM    = 8      # Effective batch = 16
MAX_SEQ_LEN   = 4096

print("Config set.")

## Cell 2: Baseline Sample

Run unmodified Nano on ~500 stratified problems to estimate per-category accuracy.
~15 min on A100. Tells us where to focus LoRA training.

In [ ]:
# ── Load problems and sample ───────────────────────────────────────
problems = load_problems(TRAIN_CSV)
sample = stratified_sample(problems, n_per_category=83)  # ~500 total
print(f"Sample: {len(sample)} problems")

In [ ]:
# ── Run baseline inference ─────────────────────────────────────────
baseline_ev = Evaluator(model_path=MODEL_PATH)  # No LoRA
baseline_summary = baseline_ev.run(sample)
Evaluator.report(baseline_summary)
baseline_ev.save_results(baseline_summary, f"{RESULTS_DIR}/baseline_results.jsonl")

In [ ]:
# ── Save baseline report to Drive ─────────────────────────────────
import json
with open(f"{DRIVE_PATH}/baseline_summary.json", "w") as f:
    json.dump({
        "accuracy": baseline_summary.accuracy,
        "total": baseline_summary.total,
        "correct": baseline_summary.correct,
        "per_category": baseline_summary.per_category,
    }, f, indent=2)
print("Baseline saved to Drive.")

# Clean up model from GPU memory before training
del baseline_ev._llm
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")

## Cell 3: Train LoRA

SFT on formatted training data using PEFT + TRL SFTTrainer.

In [ ]:
# ── Load formatted training data ───────────────────────────────────
from datasets import load_dataset

train_ds = load_dataset("json", data_files=TRAIN_JSONL, split="train")
val_ds   = load_dataset("json", data_files=VAL_JSONL,   split="train")

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print("Sample prompt (first 200 chars):")
print(train_ds[0]["prompt"][:200])
print("Sample response (first 200 chars):")
print(train_ds[0]["response"][:200])

In [ ]:
# ── Load model and apply LoRA ──────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in BF16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA applied.")

In [ ]:
# ── Format data for SFTTrainer ─────────────────────────────────────
def format_for_sft(example):
    """Combine prompt + response into a single text field for SFT."""
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_ds = train_ds.map(format_for_sft)
val_ds   = val_ds.map(format_for_sft)
print("Data formatted for SFT.")
print("Sample text (first 300 chars):")
print(train_ds[0]["text"][:300])

In [ ]:
# ── Train ──────────────────────────────────────────────────────────
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete.")

In [ ]:
# ── Save LoRA adapter ──────────────────────────────────────────────
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Copy to Drive for persistence
import shutil
shutil.copytree(ADAPTER_DIR, f"{DRIVE_PATH}/lora_adapter_v1", dirs_exist_ok=True)
print(f"Adapter saved to Drive: {DRIVE_PATH}/lora_adapter_v1")

# Free training resources
del trainer, model
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")

## Cell 4: Evaluate

Run the trained LoRA on the validation split and compare to baseline.

In [ ]:
# ── Load val problems ──────────────────────────────────────────────
import json
val_ids = set()
with open(VAL_JSONL) as f:
    for line in f:
        val_ids.add(json.loads(line)["id"])

all_problems = load_problems(TRAIN_CSV)
val_problems = [p for p in all_problems if p.id in val_ids]
print(f"Val set: {len(val_problems)} problems")

In [ ]:
# ── Evaluate with LoRA ─────────────────────────────────────────────
lora_ev = Evaluator(model_path=MODEL_PATH, lora_path=ADAPTER_DIR)
lora_summary = lora_ev.run(val_problems)
Evaluator.report(lora_summary)
lora_ev.save_results(lora_summary, f"{RESULTS_DIR}/lora_v1_results.jsonl")

In [ ]:
# ── Compare baseline vs LoRA ───────────────────────────────────────
print("\n=== Baseline vs LoRA Comparison ===")
print(f"{'Category':<25} {'Baseline':>10} {'LoRA v1':>10} {'Delta':>8}")
print("-" * 57)

base_cats = baseline_summary.per_category
lora_cats = lora_summary.per_category

for cat in sorted(lora_cats):
    base_acc = base_cats.get(cat, {}).get("accuracy", 0.0)
    lora_acc = lora_cats[cat]["accuracy"]
    delta = lora_acc - base_acc
    sign = "+" if delta >= 0 else ""
    print(f"{cat:<25} {base_acc*100:>9.1f}% {lora_acc*100:>9.1f}% {sign}{delta*100:>7.1f}%")

print("-" * 57)
base_total = baseline_summary.accuracy
lora_total = lora_summary.accuracy
delta_total = lora_total - base_total
sign = "+" if delta_total >= 0 else ""
print(f"{'TOTAL':<25} {base_total*100:>9.1f}% {lora_total*100:>9.1f}% {sign}{delta_total*100:>7.1f}%")

## Cell 5: Package for Submission

The competition expects a zip file containing the LoRA adapter files.

In [ ]:
# ── Package adapter as zip ─────────────────────────────────────────
import zipfile, os, glob

SUBMISSION_ZIP = "/content/submission.zip"

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for filepath in glob.glob(f"{ADAPTER_DIR}/**/*", recursive=True):
        if os.path.isfile(filepath):
            arcname = os.path.relpath(filepath, ADAPTER_DIR)
            zf.write(filepath, arcname)

zip_size = os.path.getsize(SUBMISSION_ZIP) / 1e6
print(f"submission.zip created: {zip_size:.1f} MB")

# Verify adapter_config.json is present (required by metric)
with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    names = zf.namelist()
    assert "adapter_config.json" in names, "adapter_config.json missing from zip!"
    print(f"Files in zip: {names}")
    print("adapter_config.json present ✓")

# Copy to Drive
import shutil
shutil.copy(SUBMISSION_ZIP, f"{DRIVE_PATH}/submission_v1.zip")
print(f"Saved to Drive: {DRIVE_PATH}/submission_v1.zip")

In [ ]:
# ── Download to local machine ──────────────────────────────────────
from google.colab import files
files.download(SUBMISSION_ZIP)
print("Download started. Submit submission.zip to Kaggle.")